In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import (
    load_sales, load_web_traffic, 
    load_promotions, load_order_items, 
    load_orders
)
from src.features import add_time_features, add_lag_features, list_feature_columns, one_hot_encode
from src.pipelines import fit_and_predict, train_validate_models
from src.models import SklearnRegressorConfig, SklearnRegressorWrapper
import pandas as pd


DATA_ROOT = project_root / "data/datathon-2026-round-1"

Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [ ]:
TRAIN_RANGE = ("2013-01-01", "2022-12-31")
PREDICT_RANGE = ("2023-01-01", "2024-07-01")
df_sales = load_sales(data_root=DATA_ROOT)

In [ ]:
TRAFFIC_FEATURES_RAW = ["date", "sessions", "unique_visitors"]
SALE_FEATURES_RAW = ["date", "Revenue", "COGS"]

TRAFFIC_ENGINEER = ["date", "returning_rate"]

In [ ]:
df = pd.DataFrame(
    {"date": pd.date_range(start=TRAIN_RANGE[0], end=PREDICT_RANGE[1], freq="D")}
)
df = df.merge(df_sales[SALE_FEATURES_RAW], on="date", how="left")
# add feature year_after_shock = max(0, year - 2019)
# add feature shock_period = 1 if date is between 2019-03-01 and 2022-12-31, else 0

df["year_after_shock"] = (df["date"].dt.year - 2019).clip(lower=0)
df["shock_period"] = ((df["date"] >= "2019-03-01") & (df["date"] <= "2022-12-31")).astype(int)


In [ ]:
df = add_time_features(df, date_col="date")
categorical_cols = ["date_of_week", "month", "day", "is_weekend",
                    "shock_period"]
print(df.head())

df = one_hot_encode(df, categorical_cols)

        date     Revenue        COGS  year_after_shock  shock_period  year  \
0 2013-01-01  5304546.99  4156070.20                 0             0  2013   
1 2013-01-02  1606940.44  1237497.84                 0             0  2013   
2 2013-01-03  2281680.01  1832133.02                 0             0  2013   
3 2013-01-04  2376895.46  1940747.07                 0             0  2013   
4 2013-01-05  2509462.77  1977027.71                 0             0  2013   

   month  day  day_of_week  day_of_year  week_of_year  is_month_end  \
0      1    1            1            1             1             0   
1      1    2            2            2             1             0   
2      1    3            3            3             1             0   
3      1    4            4            4             1             0   
4      1    5            5            5             1             0   

   is_month_start  is_weekend  
0               1           0  
1               0           0  
2       

In [ ]:
model_config = SklearnRegressorConfig(
    model_type="lightgbm",
)

result = train_validate_models(
    df=df,
    model_config=model_config,
    train_range=TRAIN_RANGE,
    predict_range=PREDICT_RANGE,
)

# submission = fit_and_predict(
#     df=df,
#     model_config=model_config,
#     train_range=TRAIN_RANGE,
#     predict_range=PREDICT_RANGE,
# )
# submission.head()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001502 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 435
[LightGBM] [Info] Number of data points in the train set: 3652, number of used features: 54
[LightGBM] [Info] Start training from score 4295996.395203
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000690 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 435
[LightGBM] [Info] Number of data points in the train set: 3652, number of used features: 54
[LightGBM] [Info] Start training from score 1.157164


,Date,Revenue,COGS
0,2023-01-01,3.137948e+06,3.175742e+06
1,2023-01-02,1.773831e+06,1.529886e+06
2,2023-01-03,9.928348e+05,8.494779e+05
3,2023-01-04,9.141336e+05,7.650498e+05
4,2023-01-05,9.975186e+05,8.450789e+05


In [ ]:
submission_name = "shocku"
submission.to_csv(project_root / f"submissions/{submission_name}.csv", index=False)